# F, G, H. 평가 종합 · 보고서 생성 · 보고서 검증

| | F. 평가 종합 | G. 보고서 생성 | H. 보고서 검증 |
|---|---|---|---|
| **담당** | LLM (검색 없음) | LLM (검색 없음) | 규칙 기반 (LLM 없음) |
| **선행 노드** | C, D, E | F (또는 H 재진입) | G |
| **출력** | `synthesis` | `final_report` | `validation_result`, `retry_count` |

셋이 순서대로 이어지고(F→G→H) H가 실패하면 G로 되돌아가는 유일한 루프라 한 노트북에 같이 둔다.
H는 LLM을 안 쓰는 규칙 기반 노드라는 게 포인트 — 챕터 존재 여부·서열 표현 포함 여부를 문자열 검사로만 판단한다.

이 노트북 끝에서 만든 것들은 `src/nodes_fgh.py`로 저장된다.

In [ ]:
import sys
sys.path.insert(0, "..")

from src import config, prompts
from src.schemas import Label, ValidationResult

## 1. F. 평가 종합 — 프롬프트 확인

In [ ]:
print(prompts.SYNTHESIS_PROMPT)

## 2. F. 노드 함수 정의

`market_eval`/`stakeholder_eval`/`domain_eval`은 C/D/E가 각자 쓴 State 키에서 읽고,
TRL은 `tech_research`(B가 씀) 안에 있어서 여기서 따로 뽑아낸다 — TRL 전담 에이전트가 없기 때문(2-1절).

In [ ]:
def make_node_f(llm):
    """F. 평가 종합. 4관점(시장/이해관계자/도메인/TRL) 라벨을 모아 비교한다.

    labels 스키마를 여기서 고정해 만든다 — 공유 파일 schemas.py 의
    Synthesis.labels 가 dict[str, str](자유 형식 object)라 두 가지가 깨진다.
      1) OpenAI 기본 strict 구조화 출력(json_schema)이 400 으로 거부한다.
         "'required' ... must include every key in properties"
      2) method="function_calling" 으로 우회하면 호출마다 모양이 달라진다.
         실측: 1회차는 한국어 키 dict, 2회차는 list 가 돌아와 ValidationError.
    schemas.py 를 건드리지 않으려고 create_model 로 고정 스키마를 만들어
    LLM 에 넘기고, State 에는 기존과 똑같은 dict 모양으로 되돌려 넣는다.
    (근본 해결은 schemas.py 의 labels 를 고정 필드로 바꾸는 것 — 조 합의 필요)
    클래스 문으로 안 쓰는 이유: Jupyter 셀에서 정의한 클래스는
    inspect.getsource 가 못 읽어 '파일로 저장' 셀이 깨진다.
    """
    from pydantic import create_model

    labels_model = create_model(
        "SynthesisLabels",
        market=(Label, ...),
        stakeholder=(Label, ...),
        domain=(Label, ...),
        trl=(Label, ...),
    )
    strict_model = create_model(
        "SynthesisStrict",
        labels=(labels_model, ...),
        conflicts=(list[str], ...),
        reasoning=(str, ...),
    )
    structured_llm = llm.with_structured_output(strict_model)

    def node_f_synthesis(state):
        tech_research = state.get("tech_research", {})
        trl_eval = {
            name: r.get("trl_assessment", {}) for name, r in tech_research.items()
        }
        prompt = prompts.SYNTHESIS_PROMPT.format(
            market_eval=state.get("market_eval", {}),
            stakeholder_eval=state.get("stakeholder_eval", {}),
            domain_eval=state.get("domain_eval", {}),
            trl_eval=trl_eval,
        )
        result = structured_llm.invoke(prompt)
        # State 모양은 기존 Synthesis.model_dump() 와 동일하게 유지한다.
        return {
            "synthesis": {
                "labels": result.labels.model_dump(),
                "conflicts": result.conflicts,
                "reasoning": result.reasoning,
            }
        }

    return node_f_synthesis

## 3. G. 보고서 생성 — 프롬프트 확인

4개로 나뉜 `*_references`를 여기서 하나로 합친다(reducer가 아니라 그냥 리스트 덧셈).
분석 배경은 State가 아니라 `config.ANALYSIS_BACKGROUND` 고정 문단을 그대로 쓴다.

In [ ]:
print(prompts.REPORT_PROMPT)

In [ ]:
def make_node_g(llm):
    """G. 보고서 생성. H가 무효 판정을 내리면 재진입해서 revision_note를 받는다."""
    def node_g_report(state):
        validation = state.get("validation_result")
        revision_note = ""
        if validation and not validation.get("is_valid", True):
            items = validation.get("missing_items", [])
            absent = [i for i in items if not i.startswith("서열 표현")]
            violations = [i for i in items if i.startswith("서열 표현")]
            parts = []
            if absent:
                parts.append(
                    f"다음 장이 빠졌다: {', '.join(absent)}. 이번엔 반드시 포함하라."
                )
            if violations:
                parts.append(
                    f"다음이 본문에 들어 있다: {', '.join(violations)}. "
                    "해당 표현을 지우거나 중립 서술로 바꿔라. "
                    "단 금칙어를 설명하는 문장 자체도 쓰지 말 것."
                )
            revision_note = "[재작성 지시] " + " ".join(parts)

        all_references = (
            state.get("tech_references", [])
            + state.get("market_references", [])
            + state.get("stakeholder_references", [])
            + state.get("domain_references", [])
        )

        # F가 만든 라벨만 넘기면 E가 뽑은 GB·%p·ms·W 수치와 각 관점의 notes가
        # 보고서에 도달하지 않는다. REPORT_PROMPT(공유 파일)를 고치지 않고
        # {synthesis} 슬롯에 종합과 원자료를 함께 실어 보낸다.
        perspective_block = {
            "관점 간 종합(F)": state.get("synthesis", {}),
            "시장성 원자료(C)": state.get("market_eval", {}),
            "이해관계자 원자료(D)": state.get("stakeholder_eval", {}),
            "도메인 원자료(E)": state.get("domain_eval", {}),
        }

        prompt = prompts.REPORT_PROMPT.format(
            analysis_background=config.ANALYSIS_BACKGROUND,
            selected_technologies=state.get("selected_technologies", {}),
            tech_research=state.get("tech_research", {}),
            synthesis=perspective_block,
            references=all_references,
            revision_note=revision_note,
        )
        report_text = llm.invoke(prompt).content
        return {"final_report": report_text}

    return node_g_report

## 4. H. 보고서 검증 — 규칙 정의

LLM을 안 부른다. `REQUIRED_CHAPTERS`가 전부 본문에 있는지, `FORBIDDEN_WORDS`(서열 표현)가 안 섞였는지만 문자열로 검사한다.
`retry_count`가 `config.MAX_RETRY_H`(2)를 넘으면 강제로 통과시키되 `forced_pass=True`로 표시한다 — G가 이걸 보고 「한계점」 장에 뭐가 빠졌는지 적는다.

In [ ]:
REQUIRED_CHAPTERS = ["SUMMARY", "시장", "이해관계자", "도메인", "REFERENCE"]
FORBIDDEN_WORDS = ["우수", "우월", "우위", "권장", "추천"]

# 서열어 검사를 적용할 장. 설계 2절이 정한 범위다.
# 「기술 선정」과 「한계점」에는 비교 문장이 설계상 들어가고,
# REPORT_PROMPT 자체가 G에게 "우수·우월·우위·권장·추천을 쓰지 않는다"고
# 지시하므로, 보고서 전체를 통짜로 검사하면 그 지시를 옮겨 적은 한 줄에
# 검사가 걸려 재작성 루프를 헛돌게 된다.
RANKING_CHECK_CHAPTERS = ["관점별 평가", "시사점"]


def _split_chapters(report):
    """마크다운 헤더 기준으로 보고서를 {장 제목: 본문}으로 자른다."""
    import re

    chapters = {}
    title, buf = "(머리말)", []
    for line in report.split("\n"):
        m = re.match(r"^\s{0,3}#{1,6}\s+(.+?)\s*$", line)
        if m:
            chapters[title] = "\n".join(buf)
            title, buf = m.group(1), []
        else:
            buf.append(line)
    chapters[title] = "\n".join(buf)
    return chapters


def node_h_validate(state):
    report = state.get("final_report", "")
    retry_count = state.get("retry_count", 0)
    chapters = _split_chapters(report)
    titles = list(chapters.keys())

    # 장 제목에서 찾는다. 본문에 "시장"이라는 낱말이 있다고 장이 있는 건 아니다.
    missing = [c for c in REQUIRED_CHAPTERS if not any(c in t for t in titles)]

    # 서열어는 지정한 장 안에서만 본다.
    scoped = "\n".join(
        body
        for t, body in chapters.items()
        if any(k in t for k in RANKING_CHECK_CHAPTERS)
    )
    forbidden_found = [w for w in FORBIDDEN_WORDS if w in scoped]
    if forbidden_found:
        missing.append(f"서열 표현 발견: {forbidden_found}")

    if not missing:
        result = ValidationResult(is_valid=True, missing_items=[], forced_pass=False)
        return {"validation_result": result.model_dump(), "retry_count": retry_count}

    new_retry_count = retry_count + 1
    if new_retry_count > config.MAX_RETRY_H:
        # 상한 도달. 통과시키되 검증 실패 사실을 보고서에 남긴다.
        # 이게 없으면 검증에 실패한 보고서가 통과 표시로 제출물이 되고,
        # 콘솔 print 말고는 어디에도 흔적이 남지 않는다.
        result = ValidationResult(is_valid=True, missing_items=missing, forced_pass=True)
        return {
            "validation_result": result.model_dump(),
            "retry_count": new_retry_count,
            "final_report": _append_audit_note(report, missing, new_retry_count),
        }

    result = ValidationResult(is_valid=False, missing_items=missing, forced_pass=False)
    return {"validation_result": result.model_dump(), "retry_count": new_retry_count}


def _append_audit_note(report, missing, retry_count):
    """forced_pass 시 검증 기록을 「한계점」 장 뒤(REFERENCE 앞)에 끼워 넣는다."""
    import re

    note = (
        "\n\n### 보고서 검증 기록 (자동 생성)\n\n"
        f"검증 노드(H)가 재작성 상한({config.MAX_RETRY_H}회)에 도달해 "
        f"아래 항목을 충족하지 못한 채 통과 처리했다. 재작성 시도 {retry_count}회.\n\n"
        + "\n".join(f"- {m}" for m in missing)
        + "\n"
    )
    for line in report.split("\n"):
        if re.match(r"^\s{0,3}#{1,6}\s+.*REFERENCE", line):
            return report.replace(line, note.strip() + "\n\n" + line, 1)
    return report + note


def route_after_h(state):
    """H 다음 조건부 엣지. graph.py의 add_conditional_edges가 이 함수를 쓴다."""
    validation = state.get("validation_result", {})
    if validation.get("is_valid", False):
        return "END"
    return "G"

## 5. 배선 테스트 — API 키 없이

H의 재시도 루프(최초 1회 + 재시도 2회 = 총 3회, 상한 도달 시 강제통과)가 제대로 도는지까지 확인한다.

In [ ]:
class FakeStructuredLLM:
    def __init__(self, output):
        self.output = output
    def invoke(self, prompt):
        return self.output

class FakeMessage:
    def __init__(self, content):
        self.content = content

class FakeLLM_F:
    def with_structured_output(self, schema_cls):
        return FakeStructuredLLM(Synthesis(
            labels={"market": "조건 의존", "stakeholder": "조건 의존", "domain": "조건 의존", "trl": "판단보류"},
            conflicts=[], reasoning="가짜 이유",
        ))

# F 테스트
node_f = make_node_f(FakeLLM_F())
f_result = node_f({})
assert "conflicts" in f_result["synthesis"]
print("F 배선 OK")

# G, H 테스트: G가 매번 다른 보고서를 내도록 만들어서 H->G 재시도 루프를 직접 확인
call_count = {"n": 0}
class FakeLLM_G:
    def invoke(self, prompt):
        call_count["n"] += 1
        if call_count["n"] == 1:
            return FakeMessage("# SUMMARY\n내용\n# 시장\n내용\n# 이해관계자\n내용\n# REFERENCE\n내용")  # 도메인 누락
        return FakeMessage("# SUMMARY\n내용\n# 시장\n내용\n# 이해관계자\n내용\n# 도메인\n내용\n# REFERENCE\n내용")

node_g = make_node_g(FakeLLM_G())

state = {}
state.update(node_g(state))
state.update(node_h_validate(state))
assert route_after_h(state) == "G"  # 도메인 누락이라 재시도
print("1차 검증:", state["validation_result"])

state.update(node_g(state))
state.update(node_h_validate(state))
assert route_after_h(state) == "END"  # 이번엔 통과
print("2차 검증:", state["validation_result"])
print("G/H 배선 OK, retry_count =", state["retry_count"])

## 6. 실제 LLM 테스트 (F만 — G/H는 최종 통합 노트북에서 실제 데이터로 보는 게 더 의미 있다)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

if os.environ.get("OPENAI_API_KEY"):
    from langchain.chat_models import init_chat_model

    real_llm = init_chat_model(config.LLM_MODEL, model_provider=config.LLM_PROVIDER, temperature=0)
    node_f_real = make_node_f(real_llm)
    sample_state = {
        "tech_research": {"TurboQuant": {"trl_assessment": {"trl_ondevice": 4}}, "InfiniGen": {"trl_assessment": {"trl_ondevice": 3}}},
        "market_eval": {"label": "조건 의존"},
        "stakeholder_eval": {"label": "조건 의존"},
        "domain_eval": {"label": "조건 의존"},
    }
    print(node_f_real(sample_state)["synthesis"])
else:
    print("API 키 없음 - 이 셀은 건너뜀.")

## 7. 파일로 저장

In [ ]:
import inspect

q3 = chr(34) * 3
with open("../src/nodes_fgh.py", "w", encoding="utf-8") as f:
    f.write(q3 + "F, G, H. 평가종합/보고서생성/보고서검증 노드 - 04_agent_FGH_synthesis_report_validate.ipynb에서 생성됨.\n")
    f.write("이 파일을 직접 고치지 말고, 노트북에서 고친 뒤 저장 셀을 다시 실행할 것." + q3 + "\n\n")
    f.write("from src import config, prompts\n")
    f.write("from src.schemas import Label, ValidationResult\n\n\n")
    f.write(inspect.getsource(make_node_f))
    f.write("\n\n")
    f.write(inspect.getsource(make_node_g))
    f.write("\n\n")
    f.write(f'REQUIRED_CHAPTERS = {REQUIRED_CHAPTERS!r}\n')
    f.write(f'FORBIDDEN_WORDS = {FORBIDDEN_WORDS!r}\n')
    f.write(f'RANKING_CHECK_CHAPTERS = {RANKING_CHECK_CHAPTERS!r}\n\n\n')
    f.write(inspect.getsource(_split_chapters))
    f.write("\n\n")
    f.write(inspect.getsource(node_h_validate))
    f.write("\n\n")
    f.write(inspect.getsource(_append_audit_note))
    f.write("\n\n")
    f.write(inspect.getsource(route_after_h))
    f.write("\n")

print("src/nodes_fgh.py 저장 완료")